# Reasoning vs. Plain RAG -- the real evaluation

*Level 8 — Reasoning Strategies*

## Objective

The whole point of this level: not just "does each strategy run," but a real, measured answer to
whether the extra structure (and extra LLM calls) that Tree-of-Thought and Graph-of-Thoughts add
over plain Chain-of-Thought is actually worth it -- on real StrategyQA questions, checked against
real ground truth, with real LLM-call costs counted, not assumed.


In [1]:
import json
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))

from reasoning_eval.metrics import compare_strategies

eval_run = json.loads((LEVEL_DIR / "data" / "cache" / "eval_run.json").read_text())
print("Questions evaluated:", len(eval_run["qids"]))
for qid in eval_run["qids"]:
    print(" ", qid, "-> ground truth:", eval_run["ground_truth"][qid])

Questions evaluated: 8
  1c56958c04a98ccd5d1e -> ground truth: False
  c5423254bd666acc8a9c -> ground truth: False
  ee6e0bcf7be93da46dcc -> ground truth: False
  646d3503f3a3939e2e63 -> ground truth: True
  589eb85285b438a0c59f -> ground truth: False
  a9d639b7f43108848d99 -> ground truth: False
  3bba4ea27500361cb141 -> ground truth: False
  810d95d42fdcfbc153df -> ground truth: False


## The comparison table -- accuracy AND cost, side by side

This is the actual real run (8 real StrategyQA questions, same retrieved evidence given to every
strategy for a fair comparison), computed with `reasoning_eval/metrics.py`'s `compare_strategies`
-- the exact same function a future, larger run would use.

In [2]:
comparison = eval_run["comparison"]
avg_time = eval_run["avg_time_seconds"]

print(f"{'strategy':6s} {'accuracy':>9s} {'avg calls':>10s} {'avg time':>9s}")
for name in ["cot", "tot", "got", "hgot"]:
    c = comparison[name]
    print(f"{name:6s} {c['accuracy']:9.3f} {c['avg_llm_calls_per_question']:10.2f} {avg_time[name]:8.1f}s")

strategy  accuracy  avg calls  avg time
cot        1.000       1.00     14.2s
tot        0.750       5.00      9.6s
got        0.500       7.00     23.3s
hgot       0.750       5.00     14.5s


## What I observed

**The simplest, cheapest strategy won.** Chain-of-Thought scored 1.000 accuracy at 1 LLM call per
question; every more elaborate strategy scored lower while costing 5-7x more calls:

| strategy | accuracy | avg LLM calls | avg time |
|---|---|---|---|
| CoT | **1.000** | **1.00** | 14.2s |
| ToT | 0.750 | 5.00 | 9.6s |
| GoT | **0.500** | **7.00** | 23.3s |
| HGoT | 0.750 | 5.00 | 14.5s |

This is a small sample (8 real questions, imbalanced toward `False` by chance with this seed --
worth naming plainly rather than hiding) but the pattern is not noise: it is explained by a
specific, traceable, reproducible mechanism, not a coincidence -- see the Mount Fuji case directly
below, and `02_tree_of_thought.ipynb` / `03_graph_of_thought.ipynb` for the full traces.
GoT scoring the *lowest* of all four, at the *highest* cost, is the most striking single number in
this table: more machinery, applied on top of the same small model doing the judging, did not
average out that model's own unreliability -- it compounded it.

## The clearest individual case: Mount Fuji

Already traced in detail in `02_tree_of_thought.ipynb` and `03_graph_of_thought.ipynb` -- shown
here again specifically side by side, all four strategies on the exact same question.

In [3]:
fuji_qid = "646d3503f3a3939e2e63"
print("Question:", [r["reasoning"] for r in [eval_run["raw_results"]["cot"][fuji_qid]]][0][:0] or "Would the top of Mount Fuji stick out of the Sea of Japan?")
print("Ground truth:", eval_run["ground_truth"][fuji_qid])
print()
for strategy in ["cot", "tot", "got", "hgot"]:
    r = eval_run["raw_results"][strategy][fuji_qid]
    print(f"{strategy:5s}: answer={r['answer']!s:5s} calls={r['llm_calls']}")
    if "best_path" in r and r["best_path"]:
        print(f"       path: {r['best_path']}")
    if "sub_questions" in r:
        print(f"       sub-questions: {r['sub_questions']}")

Question: Would the top of Mount Fuji stick out of the Sea of Japan?
Ground truth: True

cot  : answer=True  calls=1
tot  : answer=False calls=5
       path: ["The Sea of Japan's maximum depth is greater than Mount Fuji's height"]
got  : answer=False calls=7
       path: ["The Sea of Japan's maximum depth is greater than Mount Fuji's height"]
hgot : answer=False calls=5
       sub-questions: ['Is the top of Mount Fuji higher than the highest point in the Sea of Japan?', 'Is Mount Fuji located on land or on water?', 'Is the Sea of Japan a body of saltwater?']


## What I observed (Mount Fuji)

All three of the more elaborate strategies (ToT, GoT, HGoT) got this question wrong; only plain
CoT got it right. ToT and GoT both settled on the identical wrong reasoning step -- *"The Sea of
Japan's maximum depth is greater than Mount Fuji's height"* -- which is backwards on the real
numbers (Mount Fuji: ~12,389ft; the sea's maximum depth: ~12,276ft). The state evaluator scored
this specific, checkable, wrong claim with high confidence in both runs. HGoT's decomposition
avoided that exact wrong claim but still reached the wrong final answer, from a different path.
Chain-of-Thought, forced to write out the unit conversion and comparison explicitly in one pass,
got the arithmetic right. This is the one case, traced start to finish, that explains the table
above -- not an assumption about why the numbers came out this way, but the actual mechanism,
observed directly.

## Weighing this against Level 7's own measured cost

[Level 7's load test](../../07-production-rag/load-testing/scenarios.md) found a single generation
call already costs 3-6 seconds cold, and 16-40 seconds under just 5 concurrent users, on this
repo's CPU-bound Ollama setup. Multiplying that by ToT's or GoT's real average call count (5-7x a
single CoT call, measured above) is not a hypothetical concern -- it is directly, multiplicatively
worse under the exact load conditions Level 7 already measured as this repo's binding bottleneck.

## Common Failure Modes -- what this level actually found

- **The more expensive strategies did not win here.** This is a genuinely important, real result,
  not a hedge: on this real 8-question sample, plain Chain-of-Thought was the cheapest option
  *and* matched or beat Tree-of-Thought, Graph-of-Thoughts, and HGoT on accuracy. This is the same
  lesson [Level 4](../../04-adaptive-rag/README.md#evaluation--what-actually-happened) already
  found with multi-hop decomposition losing to plain retrieval, now confirmed again on an entirely
  different axis (reasoning strategy, not retrieval strategy) with an entirely different dataset.
- **The state evaluator's own error rate is not a footnote -- it visibly changed the answer** on
  the Mount Fuji case: a specific, checkable numeric claim was scored confidently *and wrong*, and
  the search followed it. Branching and scoring only help if the scoring itself is trustworthy;
  measured here, with a 3B local model doing the scoring, it was not, at least once, concretely.
- **Small sample size, stated plainly**: 8 questions is enough to *see* a real, reproducible
  effect (the Mount Fuji case is not noise -- it is a specific, traceable, checkable mistake), but
  not enough to claim a precise accuracy percentage generalizes. The honest claim this level
  supports is narrower and still real: on real 3B-local-model StrategyQA reasoning, more
  structure did not clearly buy more correctness, and cost 5-7x more to find out.
- **A judge (evaluator) sharing the same underlying model as the reasoner does not average out
  its own mistakes -- it can compound them.** GoT's extra aggregation step, layered on top of
  ToT's already-unreliable evaluator, scored the *lowest* accuracy of all four strategies in this
  run (0.500 vs. ToT's 0.750), at the highest cost (7 calls/question average, one single question
  took over 100 seconds) -- more machinery did not average away the underlying model's own
  reliability limit; it added more places for that same limit to bite.